# NCHU Rental — Colab Bi-Encoder Training (T2)

**使用前請確認**：Runtime → Change runtime type → **T4 GPU**

訓練向量檢索用的 **bi-encoder**（對比學習 / InfoNCE），CE 同源 base = `hfl/rbt6`。
與 `colab_train.ipynb`（cross-encoder）獨立，**不互相覆寫**：bi-encoder 存到 `rbt6_bi_encoder/`。

流程：
1. 確認 GPU
2. Clone / pull repo
3. 安裝依賴（**不需降級 torch**；T3 export 用 `dynamo=False` legacy tracer）
4. 確認訓練資料
5. 訓練 bi-encoder（GPU 約數分鐘 ~ 十幾分鐘，視 epoch）
6. 備份權重到 Drive

> **下一步 = T3**：用 `pipeline/model_training/exporter.py`（`dynamo=False`, opset 15）
> 匯出 **query 端**（input_ids+attention_mask → mean-pool → L2-norm embedding）並量化，
> 放 `frontend/models/bi_encoder_dir/`。本 notebook **只訓練、不匯出**。

## Cell 0 — 確認 GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('⚠️  沒有 GPU！請到 Runtime → Change runtime type → T4 GPU 再重跑')

## Cell 1 — Clone / Pull Repo

In [ ]:
import os

REPO_URL = "https://github.com/eric20041027/Renting-recommendation-ONNX.git"
REPO_DIR = "/content/Renting-recommendation-ONNX"

if os.path.exists(REPO_DIR):
    print('Repo 已存在，更新至最新...')
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

# bi-encoder 訓練腳本在 feat/vector-retrieval-t2-bi-encoder 分支
!git checkout feat/vector-retrieval-t2-bi-encoder 2>/dev/null || echo 'branch already checked out or merged'
!git log --oneline -3

## Cell 2 — 安裝依賴

> **不需降級 torch、不需 Restart runtime。** T3 的 ONNX export 由 `exporter.py`
> 用 `dynamo=False` 走 legacy TorchScript tracer，權重會嵌入單一 .onnx 檔（torch 2.x 相容）。

In [ ]:
!pip install -q \
    'transformers>=4.35' \
    datasets \
    'pandas>=2.0' \
    'pydantic>=2.0' \
    accelerate

import torch
print(f'✅ 安裝完成  torch={torch.__version__}（無需 Restart runtime）')

## Cell 3 — 設定超參數（bi-encoder 專屬）

In [ ]:
import os

# bi-encoder 權重存這裡（與 cross-encoder rbt6_finetuned/ 分開，不覆寫）
os.environ["BI_ENCODER_SAVED_DIR"]   = "/content/saved_models/rbt6_bi_encoder"
os.environ["MODEL_CHECKPOINT"]       = "hfl/rbt6"   # CE 同源 base
os.environ["RANDOM_SEED"]            = "42"
# 對比學習溫度（scaled cosine）；越小 softmax 越尖。0.05 = MNRL scale≈20 慣例
os.environ["BI_ENCODER_TEMPERATURE"] = "0.05"

# GPU 訓練參數（rule:batch 越大 in-batch negatives 越多 → 對比訊號越強）
EPOCHS     = 5
BATCH_SIZE = 64
LR         = 2e-5
MAX_LENGTH = 64
print('bi-encoder 超參數設定完成')

## Cell 4 — 確認訓練資料

In [ ]:
import json, os

train_path = 'data/processed/recommendation_train.json'
if not os.path.exists(train_path):
    print('⏳ 訓練資料不存在，重新生成...')
    !python3 pipeline/data_prep/generate_dataset.py
    print('✅ 生成完成')

for split in ['train', 'dev']:
    with open(f'data/processed/recommendation_{split}.json') as f:
        data = json.load(f)
    # label / is_hard 在 JSON 為字串，需 coerce
    pos  = sum(1 for s in data if str(s.get('label')) == '1')
    hard = sum(1 for s in data if str(s.get('is_hard')).lower() == 'true')
    print(f'{split:4s}: {len(data):6d} 筆  正樣本(label==1)={pos}  硬負樣本(is_hard)={hard}')

## Cell 5 — 訓練 bi-encoder

InfoNCE / MNRL：每個 anchor query 的正樣本 = label==1 房源；
in-batch negatives = 同 batch 其他 anchor 的正樣本；
hard negatives = 該 query 的 `is_hard==true` 房源（併入 batch 候選池）。

In [ ]:
!python3 -m pipeline.model_training.train_bi_encoder \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --max-length {MAX_LENGTH} \
    --output-dir $BI_ENCODER_SAVED_DIR

## Cell 6 — 備份 bi-encoder 權重到 Google Drive（強烈建議）

防止 Runtime 超時重啟後權重消失。T3（ONNX export）會從這裡讀。

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

dst = '/content/drive/MyDrive/renting_model'
os.makedirs(dst, exist_ok=True)

saved_dir = os.environ["BI_ENCODER_SAVED_DIR"]
if os.path.exists(saved_dir):
    shutil.copytree(saved_dir, os.path.join(dst, 'rbt6_bi_encoder'), dirs_exist_ok=True)
    print(f'✅ bi-encoder 權重已備份 → {dst}/rbt6_bi_encoder')
else:
    print('❌ bi-encoder 權重不存在，請確認 Cell 5 訓練完成')

## 下一步 — T3：query 端 ONNX 匯出 + 量化

本 notebook 只訓練 bi-encoder 並存權重（`save_pretrained`）。**T3** 會：

1. 用 `pipeline/model_training/exporter.py` 的 `dynamo=False` legacy tracer（opset 15）
   匯出 **query 端**：`input_ids + attention_mask → encoder → mask-aware mean-pool → L2-norm`。
2. `quantize_model.py` 量化（per_channel INT8，對齊正式環境）。
3. 放 `frontend/models/bi_encoder_dir/`，與離線預算的房源 embedding **同源**
   （同 model / mean-pool / L2-norm，內積即 cosine）。

> pool + normalize 必須在匯出的 ONNX graph 內，on-device 輸出才會與本訓練、
> 與離線房源 embedding 完全一致。詳見 `train_bi_encoder.py` 模組 docstring 的
> 「NOTE FOR T3」段落。